# PenG — AI Học Tập
Clone, install, and test PenG from Google Colab with GPU T4.

**Prerequisites:** Runtime → Change runtime type → T4 GPU

In [ ]:
!git clone https://github.com/canhcutlo/PenG.git
%cd PenG

In [ ]:
!nvidia-smi

In [ ]:
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie 2>&1 | tail -2

!pip install -r requirements-colab.txt
!pip install pyngrok nest-asyncio requests

In [ ]:
!python -m compileall app

In [ ]:
!pytest tests/ -v -m "not integration"

In [ ]:
import os
os.environ['LLM_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'
os.environ['LLM_QUANTIZE'] = 'true'
print('LLM:', os.environ['LLM_MODEL'])

In [ ]:
import torch
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    free = total - reserved
    print(f'GPU: {total:.1f}GB total, {free:.1f}GB free')
    if free < 7:
        print('WARNING: Less than 7GB free. Consider using Qwen2.5-1.5B or restart runtime.')
else:
    print('No GPU detected. LLM will run on CPU (very slow).')

In [ ]:
import subprocess, sys, time, requests, os, signal
from pyngrok import ngrok


try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
        print(f'Disconnected old tunnel: {t.public_url}')
except Exception:
    pass
ngrok.kill()
time.sleep(1)

import socket
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

if is_port_in_use(8000):
    print('Port 8000 is busy — killing old uvicorn...')
    if sys.platform == 'win32':
        os.system('taskkill /F /FI "IMAGENAME eq python.exe" /FI "PID ne {}" >nul 2>&1'.format(os.getpid()))
    else:
        os.system('fuser -k 8000/tcp 2>/dev/null')
    time.sleep(2)

proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

server_ok = False
for _ in range(30):
    time.sleep(0.5)
    try:
        r = requests.get("http://localhost:8000/api/health", timeout=1)
        if r.status_code == 200:
            print("Server started OK")
            server_ok = True
            break
    except requests.RequestException:
        pass

if not server_ok:
    print("ERROR: Server did not start. Check logs or restart runtime.")
else:
    tunnel = ngrok.connect(8000)
    url = tunnel.public_url

    print(f"\n=== PenG is running ===")
    print(f"Frontend: {url}")
    print(f"API docs: {url}/docs")
    print(f"Health:   {url}/api/health")
    print(f"========================")
print(f"\nIf you need to restart the server, use Runtime -> Restart session.")
print(f"NOTE: First query/quiz will trigger model download (~3-5 min).")

## Troubleshooting

| Vấn đề | Cách sửa |
|---|---|
| Upload file không xử lý (job stuck ở queued) | Kiểm tra `/api/jobs/{job_id}`. Nếu status=failed, xem error_message. Model OCR chưa tải → cần file thực |
| PDF không OCR được | Chạy `from app.services.ocr import ocr_pdf; await ocr_pdf('path')` để test OCR engine |
| ngrok limit 5 tunnels | **Runtime → Restart session** rồi chạy lại từ cell 1. Đừng re-run cell 6 nhiều lần |
| Frontend stuck ở loading | Mở DevTools (F12) → Network tab → xem API call nào bị lỗi. Có thể server chưa start |
| CUDA out of memory | Chuyển sang Qwen2.5-1.5B (`os.environ['LLM_MODEL'] = 'Qwen/Qwen2.5-1.5B-Instruct'`) hoặc Runtime → Restart session |
| Query trả "Không đủ dữ liệu" | Bình thường nếu tài liệu chưa index xong. Chờ job completed rồi thử lại. |
| Model download chậm | Lần đầu tải Qwen2.5-3B ~6GB từ HuggingFace. Đảm bảo kết nối internet ổn định. |
| Server không khởi động | Runtime → Restart session, chạy lại từ cell 1 |